# Part 15: Multi-Model LLM APIs in Practice & Gradio UI

> Connecting to all major frontier LLMs, comparing their responses, streaming output, and building real UIs with Gradio.

---


## 15.1 Setting Up LLM API Clients

### Environment Variables
Store API keys in a `.env` file — **never** hardcode them.


In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()  # loads .env file

OPENAI_API_KEY    = os.getenv("OPENAI_API_KEY")
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")
GOOGLE_API_KEY    = os.getenv("GOOGLE_API_KEY")
DEEPSEEK_API_KEY  = os.getenv("DEEPSEEK_API_KEY")


### OpenAI Client

In [ ]:
from openai import OpenAI

openai_client = OpenAI()  # picks up OPENAI_API_KEY automatically

response = openai_client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user",   "content": "What is the capital of France?"}
    ]
)
print(response.choices[0].message.content)


### Anthropic (Claude) Client

In [ ]:
import anthropic

claude = anthropic.Anthropic()  # picks up ANTHROPIC_API_KEY

message = claude.messages.create(
    model="claude-3-5-sonnet-20241022",
    max_tokens=1024,
    messages=[{"role": "user", "content": "What is the capital of France?"}]
)
print(message.content[0].text)


### Google Gemini Client

In [ ]:
import google.generativeai as genai

genai.configure(api_key=GOOGLE_API_KEY)
gemini = genai.GenerativeModel("gemini-2.0-flash-exp")

response = gemini.generate_content("What is the capital of France?")
print(response.text)


### DeepSeek (OpenAI-compatible endpoint)

In [ ]:
deepseek_client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com/v1"
)

response = deepseek_client.chat.completions.create(
    model="deepseek-chat",
    messages=[{"role": "user", "content": "What is the capital of France?"}]
)
print(response.choices[0].message.content)


## 15.2 Web Scraping with BeautifulSoup

Use LLMs with scraped content for context-aware answers.


In [ ]:
import requests
from bs4 import BeautifulSoup

def scrape_webpage(url: str) -> str:
    """Scrape clean text from a webpage."""
    headers = {"User-Agent": "Mozilla/5.0"}
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")
    
    # Remove scripts, styles, nav
    for tag in soup(["script", "style", "nav", "footer", "header"]):
        tag.decompose()
    
    return soup.get_text(separator="\n", strip=True)

def summarize_url(url: str) -> str:
    content = scrape_webpage(url)
    content = content[:5000]  # stay within context window
    
    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "Summarize the webpage content concisely."},
            {"role": "user",   "content": f"URL: {url}\n\nContent:\n{content}"}
        ]
    )
    return response.choices[0].message.content

# Example
# summary = summarize_url("https://example.com")


## 15.3 Streaming Responses

Streaming shows tokens as they arrive — better UX for long responses.


In [ ]:
def stream_response(prompt: str, model: str = "gpt-4o-mini"):
    """Stream LLM output token by token."""
    stream = openai_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        stream=True
    )
    for chunk in stream:
        if chunk.choices[0].delta.content is not None:
            print(chunk.choices[0].delta.content, end="", flush=True)
    print()  # newline at end

# Streaming with Anthropic Claude
def stream_claude(prompt: str):
    with claude.messages.stream(
        model="claude-3-5-sonnet-20241022",
        max_tokens=1024,
        messages=[{"role": "user", "content": prompt}]
    ) as stream:
        for text in stream.text_stream:
            print(text, end="", flush=True)
    print()


## 15.4 Multi-Model Comparison

Run the same prompt across multiple models and compare outputs.


In [ ]:
import asyncio

async def call_openai(prompt: str) -> str:
    """Async OpenAI call using thread executor."""
    import concurrent.futures
    loop = asyncio.get_event_loop()
    with concurrent.futures.ThreadPoolExecutor() as pool:
        response = await loop.run_in_executor(
            pool,
            lambda: openai_client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[{"role": "user", "content": prompt}]
            )
        )
    return response.choices[0].message.content

async def call_claude(prompt: str) -> str:
    message = await asyncio.to_thread(
        claude.messages.create,
        model="claude-3-5-haiku-20241022",
        max_tokens=512,
        messages=[{"role": "user", "content": prompt}]
    )
    return message.content[0].text

async def compare_models(prompt: str):
    """Run same prompt across models in parallel."""
    results = await asyncio.gather(
        call_openai(prompt),
        call_claude(prompt),
    )
    return {"gpt-4o-mini": results[0], "claude-haiku": results[1]}

# Usage:
# results = asyncio.run(compare_models("Explain RAG in one sentence."))


## 15.5 Building UIs with Gradio

Gradio turns Python functions into shareable web apps with zero frontend code.

### Simple Interface


In [ ]:
import gradio as gr

def chat_with_llm(user_message: str, model_choice: str) -> str:
    if model_choice == "GPT-4o-mini":
        response = openai_client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": user_message}]
        )
        return response.choices[0].message.content
    elif model_choice == "Claude Haiku":
        message = claude.messages.create(
            model="claude-3-5-haiku-20241022",
            max_tokens=512,
            messages=[{"role": "user", "content": user_message}]
        )
        return message.content[0].text

demo = gr.Interface(
    fn=chat_with_llm,
    inputs=[
        gr.Textbox(label="Your Message", lines=3),
        gr.Dropdown(choices=["GPT-4o-mini", "Claude Haiku"], label="Model", value="GPT-4o-mini")
    ],
    outputs=gr.Markdown(label="Response"),
    title="Multi-Model Chat",
    description="Compare responses from different LLMs"
)

# demo.launch()


### Streaming Gradio ChatInterface

In [ ]:
def streaming_chat(message: str, history: list):
    """Streaming response for Gradio ChatInterface."""
    messages = [{"role": "system", "content": "You are a helpful assistant."}]
    # Add history
    for user_msg, assistant_msg in history:
        messages.append({"role": "user",      "content": user_msg})
        messages.append({"role": "assistant", "content": assistant_msg})
    messages.append({"role": "user", "content": message})
    
    stream = openai_client.chat.completions.create(
        model="gpt-4o-mini", messages=messages, stream=True
    )
    response = ""
    for chunk in stream:
        if chunk.choices[0].delta.content:
            response += chunk.choices[0].delta.content
            yield response

chat_ui = gr.ChatInterface(
    fn=streaming_chat,
    title="Streaming Chatbot",
    type="tuples"
)
# chat_ui.launch()


## 15.6 Conversation History & Stateful Chat

Maintain conversation context across multiple turns.


In [ ]:
class ConversationManager:
    def __init__(self, system_prompt: str = "You are a helpful assistant."):
        self.history = [{"role": "system", "content": system_prompt}]
    
    def chat(self, user_message: str, model: str = "gpt-4o-mini") -> str:
        self.history.append({"role": "user", "content": user_message})
        response = openai_client.chat.completions.create(
            model=model, messages=self.history
        )
        assistant_reply = response.choices[0].message.content
        self.history.append({"role": "assistant", "content": assistant_reply})
        return assistant_reply
    
    def reset(self):
        self.history = [self.history[0]]  # keep system prompt

# Usage:
# conv = ConversationManager("You are an expert Python tutor.")
# print(conv.chat("What is a generator?"))
# print(conv.chat("Give me an example."))


## 15.7 Key Patterns Summary

| Pattern | Use Case |
|---------|----------|
| Single API call | One-shot tasks |
| Streaming | Long responses, better UX |
| Multi-model parallel | Comparison, ensembling |
| Conversation history | Chatbots, tutors |
| LLM-as-Judge tournament | Auto quality evaluation, model selection |
| Gradio Interface | Quick demos |
| Gradio ChatInterface | Full chat apps |

---

**Next:** [Part 16 — Tool Calling & Multi-modal](Part16_Tool_Calling_Multimodal.ipynb)

In [ ]:
# Step 5: o3-mini as judge (reasoning model — thinks before answering)
# o3-mini is ideal as judge because it uses chain-of-thought internally
judge_response = openai_client.chat.completions.create(
    model="o3-mini",
    messages=[{"role": "user", "content": judge_prompt}]
)
raw_result = judge_response.choices[0].message.content
print("Raw judge output:", raw_result)

# Step 6: Parse and display final rankings
results_dict = json.loads(raw_result)
ranks = results_dict["results"]

print("\n🏆 Final Rankings:")
for rank, competitor_num in enumerate(ranks, 1):
    model_name = competitors[int(competitor_num) - 1]
    print(f"  Rank {rank}: {model_name}")

In [ ]:
# Step 3: Format all responses for the judge
together = ""
for i, (model, answer) in enumerate(zip(competitors, answers), 1):
    together += f"# Response from competitor {i} ({model})\n\n{answer}\n\n"

# Step 4: Build the judge prompt
judge_prompt = f"""You are judging a competition between {len(competitors)} LLMs.
Each was given this question:

{question}

Evaluate each response for clarity, accuracy, depth, and strength of reasoning.
Rank them from best to worst.

Respond ONLY with JSON in this exact format:
{{"results": ["best competitor number", "second best", ...]}}

Here are the responses:

{together}

Now respond with the JSON ranking only."""

print(judge_prompt[:500], "...")  # preview

In [ ]:
messages = [{"role": "user", "content": question}]

competitors = []
answers     = []

# --- GPT-4.1-mini ---
resp = openai_client.chat.completions.create(model="gpt-4.1-mini", messages=messages)
competitors.append("gpt-4.1-mini")
answers.append(resp.choices[0].message.content)
display(Markdown(f"**GPT-4.1-mini:**\n\n{answers[-1]}"))

# --- Claude 3.7 Sonnet ---
resp = claude_client.messages.create(
    model="claude-3-7-sonnet-latest", messages=messages, max_tokens=1000
)
competitors.append("claude-3-7-sonnet")
answers.append(resp.content[0].text)
display(Markdown(f"**Claude 3.7:**\n\n{answers[-1]}"))

# --- Gemini 2.0 Flash ---
resp = gemini_client.chat.completions.create(model="gemini-2.0-flash", messages=messages)
competitors.append("gemini-2.0-flash")
answers.append(resp.choices[0].message.content)
display(Markdown(f"**Gemini 2.0:**\n\n{answers[-1]}"))

# --- DeepSeek Chat (optional — skip if no key) ---
if os.getenv("DEEPSEEK_API_KEY"):
    resp = deepseek_client.chat.completions.create(model="deepseek-chat", messages=messages)
    competitors.append("deepseek-chat")
    answers.append(resp.choices[0].message.content)
    display(Markdown(f"**DeepSeek:**\n\n{answers[-1]}"))

# --- Groq / LLaMA (optional — skip if no key) ---
if os.getenv("GROQ_API_KEY"):
    resp = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile", messages=messages
    )
    competitors.append("llama-3.3-70b (Groq)")
    answers.append(resp.choices[0].message.content)
    display(Markdown(f"**Groq LLaMA:**\n\n{answers[-1]}"))

# --- Ollama / LLaMA 3.2 (local — optional) ---
try:
    resp = ollama_client.chat.completions.create(model="llama3.2", messages=messages)
    competitors.append("llama3.2 (Ollama)")
    answers.append(resp.choices[0].message.content)
    display(Markdown(f"**Ollama llama3.2:**\n\n{answers[-1]}"))
except Exception:
    print("Ollama not running locally — skipping.")

In [ ]:
# Step 1: Let GPT generate a hard, nuanced question
question_prompt = ("Please come up with a challenging, nuanced question that I can ask "
                   "a number of LLMs to evaluate their intelligence. "
                   "Answer only with the question, no explanation.")

question = openai_client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[{"role": "user", "content": question_prompt}]
).choices[0].message.content

print("Question:", question)

In [ ]:
import os, json
from dotenv import load_dotenv
from openai import OpenAI
from anthropic import Anthropic
from IPython.display import Markdown, display

load_dotenv(override=True)

# --- Clients ---
openai_client = OpenAI()
claude_client  = Anthropic()

# All OpenAI-compatible endpoints
gemini_client   = OpenAI(api_key=os.getenv("GOOGLE_API_KEY"),
                         base_url="https://generativelanguage.googleapis.com/v1beta/openai/")
deepseek_client = OpenAI(api_key=os.getenv("DEEPSEEK_API_KEY"),
                         base_url="https://api.deepseek.com/v1")
groq_client     = OpenAI(api_key=os.getenv("GROQ_API_KEY"),
                         base_url="https://api.groq.com/openai/v1")
ollama_client   = OpenAI(api_key="ollama",
                         base_url="http://localhost:11434/v1")

## 15.8 Multi-Model Tournament with LLM-as-Judge

Run the same challenging question across **6 models**, then use a reasoning model (o3-mini) as an **impartial judge** to rank their answers.

```
Models:  GPT-4.1-mini → Claude 3.7 → Gemini 2.0 → DeepSeek → Groq/LLaMA → Ollama (local)
Judge:   o3-mini (reasoning model — thinks before judging)
```

This is a practical agentic pattern for **ensemble evaluation** and automatic quality control.